### День 1 — Базовая модель банковских счетов (усложнённый вариант)

🎯 Цель дня
Создать расширенную и абстрактную модель банковского счёта, способную служить базой для более сложных типов счетов.

📋 Требования

1. Абстрактный класс AbstractAccount
Создать абстрактный класс, содержащий:
- 🔑 уникальный идентификатор счёта
- 👤 данные владельца
- 💰 защищённый баланс
- 📊 статус счёта: активный, замороженный, закрытый
- 🔧 абстрактные методы:
  deposit(amount)
  withdraw(amount)
  get_account_info()

In [78]:
from dataclasses import dataclass
from abc import ABC, abstractmethod

'''
@dataclass cтрока-инструкция для Python: перед тем как окончательно создать
класс ниже, пропусти его через функцию dataclass
'''

@dataclass 
class AbstractAccount(ABC):
    account_id: str
    owner: str
    _balance: float
    status: str

    @abstractmethod 
    def deposit(self, amount): #положить деньги на счёт
        ...
        
    @abstractmethod
    def withdraw(self, amount): #снять деньги со счёта
        ...
        
    @abstractmethod
    def get_account_info(self): #показать выписку по счёту
        ...    

2. Класс BankAccount
Реализовать конкретный тип счёта с расширенными возможностями:
- ✅ валидация входящих данных
- 🔒 логические статусы и запрет операций при неверных статусах
- 🆔 автоматическая генерация короткого UUID при отсутствии номера счёта
- 💱 атрибут currency: RUB, USD, EUR, KZT, CNY

3. Исключения
Создать собственные классы ошибок:
- ❄️ AccountFrozenError счёт временно заблокирован
- 🚫 AccountClosedError счёта больше не существует
- ⚠️ InvalidOperationError то, что вы просите сделать, некорректно в принципе
- 💸 InsufficientFundsError денег не хватает

4. Базовые операции
Добавить проверки:
- ✅ корректность суммы
- 🔄 проверка статуса счета
- 🛡️ защита от отрицательных значений

5. Строковое представление
Метод __str__ должен показывать:
- 🏦 тип счета
- 👤 клиента
- 🔢 последние 4 цифры номера
- 📊 статус
- 💰 баланс и валюту



In [101]:
'''__init__ возьми параметры, которые пришли в BankAccount.__init__, 
и передай их дальше — пусть AbstractAccount.__init__ сделает свою обычную работу 
(сохранит их в self.account_id, self.owner и т.д.
'''
#исключения
class AccountFrozenError(Exception):
    '''Cчёт временно заблокирован'''
    pass

class AccountClosedError(Exception):
    '''Cчёт больше не существует'''
    pass

class InvalidOperationError(Exception):
    '''То, что вы просите сделать, некорректно в принципе'''
    pass

class InsufficientFundsError(Exception):
    '''Недостаточно средств на счёте'''
    pass

import uuid

'''__init__ возьми параметры, которые пришли в BankAccount.__init__, 
и передай их дальше — пусть AbstractAccount.__init__ сделает свою обычную работу 
(сохранит их в self.account_id, self.owner и т.д.
'''
import uuid

class BankAccount(AbstractAccount):
    def __init__(self, owner, _balance, status, account_id=None, currency="RUB"):
        #валидация входящих данных
        if not owner:
            raise InvalidOperationError
        if _balance < 0:
            raise InvalidOperationError
        if status not in ["active", "frozen", "closed"]:
            raise InvalidOperationError
        #автоматическая генерация короткого UUID при отсутствии номера счёта
        if account_id is None:
            account_id = uuid.uuid4().hex[:8]
        if currency not in ["RUB", "USD", "EUR", "KZT", "CNY"]:
            raise InvalidOperationError
        super().__init__(account_id, owner, _balance, status)
        self.currency = currency

    def deposit(self, amount): #положить деньги на счёт
        if amount <= 0:
            raise InvalidOperationError
        if self.status == "frozen":
            raise AccountFrozenError
        if self.status == "closed":
            raise AccountClosedError
        self._balance += amount   

    def withdraw(self, amount): #снять деньги со счёта
        if amount <= 0:
            raise InvalidOperationError
        if self.status == "frozen":
            raise AccountFrozenError
        if self.status == "closed":
            raise AccountClosedError
        if self._balance-amount<0:
            raise InsufficientFundsError
        self._balance -= amount

    def get_account_info(self):
        return {
            "owner": self.owner,
            "balance": self._balance,
            "status": self.status,
        }

    def __str__(self):
        return f"Тип счёта {type(self).__name__}, клиент {self.owner}, счёт ...{self.account_id[-4:]}, статус {self.status}, баланс {self._balance} {self.currency}"


In [102]:
acc = BankAccount("Иван", 1000.0, "active", currency="USD")
print(acc)

Тип счёта BankAccount, клиент Иван, счёт ...f312, статус active, баланс 1000.0 USD


6. Тестирование
Создать демонстрацию:
- ➕ создание активного и замороженного счёта
- 🚫 попытка операций над замороженным счётом
- ✅ валидное пополнение и снятие

In [106]:
#Создание активного счёта
active_acc = BankAccount("Иван", 1000.0, "active", currency="RUB")
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...6cd2, статус active, баланс 1000.0 RUB


In [108]:
#Создание замороженного счёта
frozen_acc = BankAccount("Мария", 500.0, "frozen", currency="USD")
print(frozen_acc)

Тип счёта BankAccount, клиент Мария, счёт ...f5e1, статус frozen, баланс 500.0 USD


In [109]:
#Попытка пополнить замороженный счёт
frozen_acc.deposit(100)

AccountFrozenError: 

In [111]:
#Попытка снять деньги с того же замороженного счёта
frozen_acc.withdraw(50)

AccountFrozenError: 

In [114]:
#Валидное пополнение счёта
active_acc.deposit(500)
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...6cd2, статус active, баланс 1500.0 RUB


In [115]:
#Валидное снятие счёта
active_acc.withdraw(200)
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...6cd2, статус active, баланс 1300.0 RUB
